# Predict side effects from molecular graphs using https://keras.io/examples/graph/mpnn-molecular-graphs/ as a tutorial

In [ ]:
import os
from pathlib import Path
import warnings

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from rdkit import Chem
from rdkit import RDLogger

warnings.filterwarnings('ignore')
RDLogger.DisableLog('rdApp.*')

# Set random seeds for reproducibility    
np.random.seed(42)
tf.random.set_seed(42)

# Prepare data path for reading the CSV file with prepared targets.
DATA_PATH = Path('..') / 'Data' / 'prepared_targets.csv'
df = pd.read_csv(DATA_PATH)

# Our targets (side effects) are from column 4 onwards, so we separate them from the rest of the data.
targets = df.iloc[:, 4:].copy()

# Remove target columns with no positive examples.
positive_counts = targets.sum(axis=0) # Count positive examples for each target
target_names = positive_counts[positive_counts > 0].index.tolist() # Keep only targets with at least one positive example

# Keep only molecules with valid SMILES strings.
valid_smiles = df['smiles'].notna() & df['smiles'].astype(str).str.strip().ne('')
df = df.loc[valid_smiles].reset_index(drop=True)
targets = targets.loc[valid_smiles, target_names].reset_index(drop=True).astype('float32')
print(f'Molecules: {len(df):,}')
print(f'Learnable side effects: {len(target_names):,}')
print(f'Positive labels: {int(targets.to_numpy().sum()):,}')

Molecules: 1,266
Learnable side effects: 5,970
Positive labels: 147,157


In [ ]:
# Featurizer class for encoding atom or bonds into fixed-length vectors. 
# Each feature is one-hot encoded, and the resulting vectors are concatenated to form a single vector for each atom or bond.
class Featurizer:
    def __init__(self, allowable_sets):
        self.dim = 0
        self.features_mapping = {}
        for name, values in allowable_sets.items():
            values = sorted(values, key=str)
            self.features_mapping[name] = dict(zip(values, range(self.dim, self.dim + len(values))))
            self.dim += len(values)

    # One-hot encoding of features into a fixed-length vector for each atom or bond 
    def encode(self, value):
        output = np.zeros(self.dim, dtype='float32') # First, all features in an atom or bond is set to 0.
        for name, mapping in self.features_mapping.items():
            feature = getattr(self, name)(value)
            if feature in mapping: # If a feature is present in the mapping, it is set to 1.0 in the output vector.
                output[mapping[feature]] = 1.0
        return output

# AtomFeaturizer class inherits the mapping from Featurizer. It retrieves atom features and encodes them into a fixed-length vector. The features include atom symbol, total valence, number of hydrogens, and hybridization state.
class AtomFeaturizer(Featurizer):
    def symbol(self, atom): return atom.GetSymbol() # Retrieves atom symbol from RDKit's atom object.
    def n_valence(self, atom): return atom.GetTotalValence() # Retrieves total valence of the atom from RDKit's atom object.
    def n_hydrogens(self, atom): return atom.GetTotalNumHs() # Retrieves total number of hydrogens attached to the atom from RDKit's atom object.
    def hybridization(self, atom): return atom.GetHybridization().name.lower() # Retrieves hybridization state of the atom from RDKit's atom object.

# BondFeaturizer class inherits the mapping from Featurizer. It retrieves bond features and encodes them into a fixed-length vector. The features include bond type and whether the bond is conjugated. 
class BondFeaturizer(Featurizer):
    def __init__(self, allowable_sets):
        super().__init__(allowable_sets)
        self.dim += 1 # Adds an extra dimension to the feature vector to account for a case where a bond is not present.

    def encode(self, bond): # Encodes bond features. If a bond is not present, it returns a vector of zeros with the last vector position set to 1.0, indicating the absence of a bond.
        if bond is None:
            output = np.zeros(self.dim, dtype='float32')
            output[-1] = 1.0
            return output
        return super().encode(bond) # If a bond is present, it calls the parent class (Featurizer) encode method to encode the bond features.

    def bond_type(self, bond): return bond.GetBondType().name.lower() # Retrieves the object's bond type, either (single, double, triple, or aromatic)
    def conjugated(self, bond): return bond.GetIsConjugated() # Retrieves a Boolean (true, false) for if the bond is conjugated or not.

# atom_featurizer and bond_featurizer are instances of AtomFeaturizer and BondFeaturizer. They are initialized with allowable sets of features for atoms and bonds. 
# The allowable sets define the possible values for each feature, which is used for one-hot encoding.
atom_featurizer = AtomFeaturizer({
    'symbol': {'B', 'Br', 'C', 'Ca', 'Cl', 'F', 'H', 'I', 'N', 'Na', 'O', 'P', 'S'},
    'n_valence': {0, 1, 2, 3, 4, 5, 6},
    'n_hydrogens': {0, 1, 2, 3, 4},
    'hybridization': {'s', 'sp', 'sp2', 'sp3'},
})
bond_featurizer = BondFeaturizer({
    'bond_type': {'single', 'double', 'triple', 'aromatic'},
    'conjugated': {True, False},
})

def molecule_from_smiles(smiles):
    return Chem.MolFromSmiles(str(smiles))

def graph_from_molecule(molecule):
    atom_features, bond_features, pair_indices = [], [], []
    for atom in molecule.GetAtoms():
        atom_features.append(atom_featurizer.encode(atom))
        pair_indices.append([atom.GetIdx(), atom.GetIdx()])
        bond_features.append(bond_featurizer.encode(None))
        for neighbor in atom.GetNeighbors():
            bond = molecule.GetBondBetweenAtoms(atom.GetIdx(), neighbor.GetIdx())
            pair_indices.append([atom.GetIdx(), neighbor.GetIdx()])
            bond_features.append(bond_featurizer.encode(bond))
    return np.asarray(atom_features), np.asarray(bond_features), np.asarray(pair_indices)

def graphs_from_smiles(smiles_list):
    atom_list, bond_list, pair_list, keep = [], [], [], []
    for index, smiles in enumerate(smiles_list):
        molecule = molecule_from_smiles(smiles)
        if molecule is None or molecule.GetNumAtoms() == 0:
            continue
        atom_features, bond_features, pair_indices = graph_from_molecule(molecule)
        atom_list.append(atom_features)
        bond_list.append(bond_features)
        pair_list.append(pair_indices)
        keep.append(index)
    return (tf.ragged.constant(atom_list, dtype=tf.float32),
            tf.ragged.constant(bond_list, dtype=tf.float32),
            tf.ragged.constant(pair_list, dtype=tf.int64), keep)

permuted = np.random.permutation(len(df))
train_end, valid_end = int(len(df) * 0.8), int(len(df) * 0.9)
splits = {'train': permuted[:train_end], 'valid': permuted[train_end:valid_end], 'test': permuted[valid_end:]}
graph_data, label_data = {}, {}
for name, indices in splits.items():
    graphs = graphs_from_smiles(df.iloc[indices]['smiles'].to_numpy())
    kept_indices = indices[graphs[3]]
    graph_data[name] = graphs[:3]
    label_data[name] = targets.iloc[kept_indices].to_numpy(dtype='float32')
    splits[name] = kept_indices
    print(f'{name}: {len(kept_indices):,} molecules')

train: 1,012 molecules
valid: 127 molecules
test: 127 molecules


In [4]:
def prepare_batch(x_batch, y_batch):
    atom_features, bond_features, pair_indices = x_batch
    num_atoms = atom_features.row_lengths()
    num_bonds = bond_features.row_lengths()
    molecule_indices = tf.range(tf.shape(num_atoms)[0])
    molecule_indicator = tf.repeat(molecule_indices, num_atoms)
    gather_indices = tf.repeat(molecule_indices[:-1], num_bonds[1:])
    increment = tf.cumsum(num_atoms[:-1])
    increment = tf.pad(tf.gather(increment, gather_indices), [(num_bonds[0], 0)])
    pair_indices = pair_indices.merge_dims(0, 1).to_tensor() + increment[:, tf.newaxis]
    atom_features = atom_features.merge_dims(0, 1).to_tensor()
    bond_features = bond_features.merge_dims(0, 1).to_tensor()
    return (atom_features, bond_features, tf.cast(pair_indices, tf.int32), molecule_indicator), y_batch

def mpnn_dataset(x, y, batch_size=32, shuffle=False):
    dataset = tf.data.Dataset.from_tensor_slices((x, y))
    if shuffle:
        dataset = dataset.shuffle(min(len(y), 1024), seed=42)
    return dataset.batch(batch_size).map(prepare_batch, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

class EdgeNetwork(layers.Layer):
    def build(self, input_shape):
        self.atom_dim, self.bond_dim = input_shape[0][-1], input_shape[1][-1]
        self.kernel = self.add_weight(shape=(self.bond_dim, self.atom_dim * self.atom_dim), initializer='glorot_uniform')
        self.bias = self.add_weight(shape=(self.atom_dim * self.atom_dim,), initializer='zeros')
    def call(self, inputs):
        atom_features, bond_features, pair_indices = inputs
        transformed_bonds = tf.reshape(tf.matmul(bond_features, self.kernel) + self.bias, (-1, self.atom_dim, self.atom_dim))
        neighbors = tf.expand_dims(tf.gather(atom_features, pair_indices[:, 1]), -1)
        messages = tf.squeeze(tf.matmul(transformed_bonds, neighbors), -1)
        return tf.math.unsorted_segment_sum(messages, pair_indices[:, 0], tf.shape(atom_features)[0])

class MessagePassing(layers.Layer):
    def __init__(self, units=64, steps=4, **kwargs):
        super().__init__(**kwargs); self.units, self.steps = units, steps
    def build(self, input_shape):
        atom_dim = input_shape[0][-1]
        self.padding = max(0, self.units - atom_dim)
        self.edge_network = EdgeNetwork()
        self.gru = layers.GRUCell(atom_dim + self.padding)
    def call(self, inputs):
        atom_features, bond_features, pair_indices = inputs
        state = tf.pad(atom_features, [(0, 0), (0, self.padding)])
        for _ in range(self.steps):
            messages = self.edge_network([state, bond_features, pair_indices])
            state, _ = self.gru(messages, [state])
        return state

class Readout(layers.Layer):
    def __init__(self, units, **kwargs):
        super().__init__(**kwargs); self.projection = layers.Dense(units, activation='relu')
    def call(self, inputs):
        atom_states, molecule_indicator = inputs
        count = tf.reduce_max(molecule_indicator) + 1
        pooled = tf.math.unsorted_segment_mean(atom_states, molecule_indicator, count)
        return self.projection(pooled)

def MPNNModel(atom_dim, bond_dim, target_dim, message_units=64, message_steps=4, dense_units=256):
    atom_features = layers.Input((atom_dim,), dtype='float32', name='atom_features')
    bond_features = layers.Input((bond_dim,), dtype='float32', name='bond_features')
    pair_indices = layers.Input((2,), dtype='int32', name='pair_indices')
    molecule_indicator = layers.Input((), dtype='int32', name='molecule_indicator')
    states = MessagePassing(message_units, message_steps)([atom_features, bond_features, pair_indices])
    embedding = Readout(message_units)([states, molecule_indicator])
    output = layers.Dense(dense_units, activation='relu')(embedding)
    output = layers.Dense(target_dim, activation='sigmoid', name='side_effect_probabilities')(output)
    return keras.Model([atom_features, bond_features, pair_indices, molecule_indicator], output)

x_train, y_train = graph_data['train'], label_data['train']
x_valid, y_valid = graph_data['valid'], label_data['valid']
x_test, y_test = graph_data['test'], label_data['test']
train_dataset = mpnn_dataset(x_train, y_train, shuffle=True)
valid_dataset = mpnn_dataset(x_valid, y_valid)
test_dataset = mpnn_dataset(x_test, y_test)
mpnn = MPNNModel(x_train[0][0][0].shape[0], x_train[1][0][0].shape[0], len(target_names))
mpnn.compile(optimizer=keras.optimizers.Adam(5e-4), loss='binary_crossentropy', metrics=[keras.metrics.AUC(name='micro_auc', multi_label=False)])
mpnn.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ atom_features       │ (None, 29)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bond_features       │ (None, 7)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pair_indices        │ (None, 2)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ message_passing     │ (None, 64)        │     57,728 │ atom_features[0]… │
│ (MessagePassing)    │                   │            │ bond_features[0]… │
│                     │                   │            │ pair_indices[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ molecule_indicator  │ (None)            │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ readout (Readout)   │ (None, 64)        │      4,160 │ message_passing[… │
│                     │                   │            │ molecule_indicat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 256)       │     16,640 │ readout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ side_effect_probab… │ (None, 5970)      │  1,534,290 │ dense_1[0][0]     │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,612,818 (6.15 MB)

 Trainable params: 1,612,818 (6.15 MB)

 Non-trainable params: 0 (0.00 B)

In [6]:
callbacks = [keras.callbacks.EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True)]
history = mpnn.fit(train_dataset, validation_data=valid_dataset, epochs=40, verbose=2, callbacks=callbacks)
eval_train_dataset = mpnn_dataset(x_train, y_train, shuffle=False)
train_probabilities = mpnn.predict(eval_train_dataset, verbose=0)
valid_probabilities = mpnn.predict(valid_dataset, verbose=0)
probabilities = mpnn.predict(test_dataset, verbose=0)

def diagnostic_metrics(labels, scores, thresholds=0.5):
    labels = np.asarray(labels, dtype='float32')
    scores = np.asarray(scores, dtype='float32')
    threshold_array = np.broadcast_to(thresholds, scores.shape[1])
    predictions = (scores >= threshold_array).astype('float32')
    true_positive = np.sum(labels * predictions)
    false_positive = np.sum((1.0 - labels) * predictions)
    false_negative = np.sum(labels * (1.0 - predictions))
    precision = true_positive / max(true_positive + false_positive, 1.0)
    recall = true_positive / max(true_positive + false_negative, 1.0)
    f1 = 2.0 * precision * recall / max(precision + recall, 1e-12)
    auc = keras.metrics.AUC(name='diagnostic_auc')
    auc.update_state(labels.ravel(), scores.ravel())
    return float(auc.result()), float(precision), float(recall), float(f1), int(predictions.sum())

print('\nOverfitting diagnostic (threshold = 0.5)')
for name, labels, scores in [
    ('train', y_train, train_probabilities),
    ('valid', y_valid, valid_probabilities),
    ('test', y_test, probabilities),
]:
    auc, precision, recall, f1, positive_count = diagnostic_metrics(labels, scores)
    print(f'  {name:5s}: AUC={auc:.4f}, precision={precision:.4f}, recall={recall:.4f}, F1={f1:.4f}, predicted positives={positive_count:,}')


Epoch 1/40
32/32 - 2s - 49ms/step - loss: 0.0613 - micro_auc: 0.9135 - val_loss: 0.0649 - val_micro_auc: 0.9055
Epoch 2/40
32/32 - 2s - 51ms/step - loss: 0.0619 - micro_auc: 0.9136 - val_loss: 0.0649 - val_micro_auc: 0.9053
Epoch 3/40
32/32 - 2s - 50ms/step - loss: 0.0619 - micro_auc: 0.9143 - val_loss: 0.0649 - val_micro_auc: 0.9051
Epoch 4/40
32/32 - 2s - 50ms/step - loss: 0.0620 - micro_auc: 0.9138 - val_loss: 0.0650 - val_micro_auc: 0.9051
Epoch 5/40
32/32 - 2s - 50ms/step - loss: 0.0622 - micro_auc: 0.9146 - val_loss: 0.0653 - val_micro_auc: 0.9042
Epoch 6/40
32/32 - 2s - 50ms/step - loss: 0.0628 - micro_auc: 0.9141 - val_loss: 0.0651 - val_micro_auc: 0.9048
Epoch 7/40
32/32 - 2s - 51ms/step - loss: 0.0621 - micro_auc: 0.9139 - val_loss: 0.0651 - val_micro_auc: 0.9054
Epoch 8/40
32/32 - 2s - 50ms/step - loss: 0.0619 - micro_auc: 0.9143 - val_loss: 0.0651 - val_micro_auc: 0.9050
Epoch 9/40
32/32 - 2s - 50ms/step - loss: 0.0627 - micro_auc: 0.9138 - val_loss: 0.0651 - val_micro_auc: